In [81]:
import pandas as pd
df = pd.read_csv("loan.csv")
df.head(5)

,age,gender,occupation,education_level,marital_status,income,credit_score,loan_status
0,32,Male,Engineer,Bachelor's,Married,85000,720,Approved
1,45,Female,Teacher,Master's,Single,62000,680,Approved
2,28,Male,Student,High School,Single,25000,590,Denied
3,51,Female,Manager,Bachelor's,Married,105000,780,Approved
4,36,Male,Accountant,Bachelor's,Married,75000,710,Approved


In [82]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   age              61 non-null     int64
 1   gender           61 non-null     str  
 2   occupation       61 non-null     str  
 3   education_level  61 non-null     str  
 4   marital_status   61 non-null     str  
 5   income           61 non-null     int64
 6   credit_score     61 non-null     int64
 7   loan_status      61 non-null     str  
dtypes: int64(3), str(5)
memory usage: 3.9 KB


In [83]:
df.describe()

,age,income,credit_score
count,61.000000,61.000000,61.000000
mean,37.081967,78983.606557,709.836066
std,8.424755,33772.025802,72.674888
min,24.000000,25000.000000,560.000000
25%,30.000000,52000.000000,650.000000
50%,36.000000,78000.000000,720.000000
75%,43.000000,98000.000000,770.000000
max,55.000000,180000.000000,830.000000


In [84]:
df["loan_status"].value_counts()

loan_status
Approved    45
Denied      16
Name: count, dtype: int64

In [85]:
df.groupby("loan_status")[["age", "income", "credit_score"]].mean()

,age,income,credit_score
loan_status,,,
Approved,40.311111,92955.555556,745.111111
Denied,28.000000,39687.500000,610.625000


In [86]:
df.groupby("loan_status")[
    ["age", "income", "credit_score"]].agg(["min", "max"])

age     income         credit_score     
            min max    min     max          min  max
loan_status                                         
Approved     28  55  55000  180000          650  830
Denied       24  33  25000   52000          560  660

1. From the dataset we can see that 74% of applicants were approved to be able to collect loans. Therefore the baseline accuracy should be 74%
2. Approved applicants have a mean income of about 93,000 while denied applicants have a mean income of about 40,000. That means approved applicants have almost double the income of those denied. Therefore making Income a major threshold for deciding whether an applicant is approved or denied.
3. Approved applicants all scored 650 or above, and denied applicants all scored 660 or below. The groups overlap only between 650 and 660, so the boundary seems to sit somewhere near there.

In [87]:
from sklearn.model_selection import train_test_split

X = df[["age", "income", "credit_score"]]   # features: the clues
y = df["loan_status"]                        # target: the answer

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
y_test.value_counts()

(45, 3) (16, 3)


loan_status
Approved    12
Denied       4
Name: count, dtype: int64

In [88]:
#X_train, X_test, y_train, y_test = train_test_split(
   # X, y, test_size=0.25, random_state=1)
#y_test.value_counts()

In [89]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(predictions)
print(y_test.values)

['Approved' 'Denied' 'Approved' 'Denied' 'Approved' 'Approved' 'Approved'
 'Denied' 'Approved' 'Approved' 'Denied' 'Approved' 'Approved' 'Approved'
 'Approved' 'Approved']
<StringArray>
['Approved',   'Denied', 'Approved',   'Denied', 'Approved', 'Approved',
 'Approved',   'Denied', 'Approved', 'Approved',   'Denied', 'Approved',
 'Approved', 'Approved', 'Approved', 'Approved']
Length: 16, dtype: str


In [90]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print(accuracy_score(y_test, predictions))
print(confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

1.0
[[12  0]
 [ 0  4]]
              precision    recall  f1-score   support

    Approved       1.00      1.00      1.00        12
      Denied       1.00      1.00      1.00         4

    accuracy                           1.00        16
   macro avg       1.00      1.00      1.00        16
weighted avg       1.00      1.00      1.00        16



In [91]:
X_age = df[["age"]]
Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X_age, y, test_size=0.25, random_state=42, stratify=y
)
age_model = LogisticRegression(max_iter=1000).fit(Xa_train, ya_train)
age_pred = age_model.predict(Xa_test)
print(confusion_matrix(ya_test, age_pred))

[[11  1]
 [ 0  4]]


In [92]:
pd.get_dummies(df[["gender"]], drop_first=True).head()

,gender_Male
0,True
1,False
2,True
3,False
4,True


In [93]:
from sklearn.metrics import accuracy_score


def score(features):
    X = pd.get_dummies(df[features], drop_first=True)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    m = LogisticRegression(max_iter=5000).fit(X_tr, y_tr)
    return accuracy_score(y_te, m.predict(X_te))


nums = ["age", "income", "credit_score"]
print("numbers only:", score(nums))
print("+ education:", score(nums + ["education_level"]))
print("+ education, gender, marital:", score(nums +
      ["education_level", "gender", "marital_status"]))
print("education only:", score(["education_level"]))

numbers only: 1.0
+ education: 1.0
+ education, gender, marital: 1.0
education only: 0.8125


In [94]:
from sklearn.tree import DecisionTreeClassifier, export_text

# use the stratified numeric split: X_train, X_test, y_train, y_test
tree = DecisionTreeClassifier(max_depth=2, random_state=42)
tree.fit(X_train, y_train)

print(export_text(tree, feature_names=list(X_train.columns)))
print(accuracy_score(y_test, tree.predict(X_test)))

|--- income <= 53500.00
|   |--- class: Denied
|--- income >  53500.00
|   |--- class: Approved

1.0


This is a model that is built to predict whether an applicant can get their loan approved. Logistic regression and a decision tree were used to build this using 61 applicants. 
The models produced a result of 100% accuracy because there is an obvious split which is about 53,500 in those who were approved and denied using the income threshold.
Using cross validation gives an average accuracy of 97%, this shows that the model isn't perfect as the single split had a lucky set of 16 appplicants but every applicant has a chance in the test split, therefore showing the error.

Wrongly approving a customer would also be worse for the bank because in this dataset, income is the major factor that decides whether a customer would be approved or denied. If a person with low income is approved it will result in bad debt as the person might not be able to pay back.

In [95]:
print((predictions == y_test.values).sum())   # how many matched
print(len(y_test))                            # out of how many

16
16


In [96]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5)
print(scores)
print(scores.mean())

[1.         0.91666667 0.91666667 1.         1.        ]
0.9666666666666666
